# Unidad 2 · Colab 3 de 3
## Archivos estructurados y el procesador de transacciones

**Objetivos de este notebook**

- Leer y escribir archivos en JSON, CSV, YAML y Parquet.
- Elegir el formato correcto según el caso de uso.
- Aplicar todo lo de esta unidad (excepciones, módulos, formatos de archivo) en un módulo procesador de transacciones con validación de datos y logs de errores.

> **Nivel:** intermedio. Este notebook integra los Colab 1 y 2 de esta unidad.

---

## 1. JSON

Formato de texto, legible, ideal para configuración y APIs (ya lo usaste en la Unidad 4).

```python
import json

datos = {'id': 1, 'monto': 150.5, 'moneda': 'ARS'}

with open('transaccion.json', 'w') as f:
    json.dump(datos, f, indent=2)

with open('transaccion.json') as f:
    cargado = json.load(f)

print(cargado)
```

Documentación oficial: [módulo json](https://docs.python.org/3/library/json.html)

In [1]:
import json

datos = {'id': 1, 'monto': 150.5, 'moneda': 'ARS'}

with open('transaccion.json', 'w') as f:
    json.dump(datos, f, indent=2)

with open('transaccion.json') as f:
    cargado = json.load(f)

print(cargado)

{'id': 1, 'monto': 150.5, 'moneda': 'ARS'}


## 2. CSV

Formato tabular de texto plano, el más simple para intercambiar datos entre sistemas (Excel, bases de datos, etc.).

```python
import csv

filas = [{'id': 1, 'monto': 150.5, 'moneda': 'ARS'}, {'id': 2, 'monto': 40.0, 'moneda': 'USD'}]

with open('transacciones.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'monto', 'moneda'])
    writer.writeheader()
    writer.writerows(filas)

with open('transacciones.csv') as f:
    reader = csv.DictReader(f)
    for fila in reader:
        print(fila)
```

Documentación oficial: [módulo csv](https://docs.python.org/3/library/csv.html)

In [2]:
import csv

filas = [{'id': 1, 'monto': 150.5, 'moneda': 'ARS'}, {'id': 2, 'monto': 40.0, 'moneda': 'USD'}]

with open('transacciones.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'monto', 'moneda'])
    writer.writeheader()
    writer.writerows(filas)

with open('transacciones.csv') as f:
    reader = csv.DictReader(f)
    for fila in reader:
        print(fila)

{'id': '1', 'monto': '150.5', 'moneda': 'ARS'}
{'id': '2', 'monto': '40.0', 'moneda': 'USD'}


### Ejercicio 1 — JSON y CSV

Escribí el código para: (1) guardar una lista de 3 transacciones (diccionarios con `id`, `monto`, `moneda`) en `transacciones.json` usando `json.dump` con `indent=2`, y (2) leerla de vuelta y confirmar que son iguales a la lista original.

<details>
<summary>💡 Ver solución</summary>

```python
import json

transacciones = [
    {'id': 1, 'monto': 100.0, 'moneda': 'ARS'},
    {'id': 2, 'monto': 200.0, 'moneda': 'USD'},
    {'id': 3, 'monto': 50.0, 'moneda': 'EUR'},
]

with open('transacciones.json', 'w') as f:
    json.dump(transacciones, f, indent=2)

with open('transacciones.json') as f:
    cargadas = json.load(f)

print(cargadas == transacciones)
```

</details>

In [3]:
import json

transacciones = [
    {'id': 1, 'monto': 100.0, 'moneda': 'ARS'},
    {'id': 2, 'monto': 200.0, 'moneda': 'USD'},
    {'id': 3, 'monto': 50.0, 'moneda': 'EUR'},
]

with open('transacciones.json', 'w') as f:
    json.dump(transacciones, f, indent=2)

with open('transacciones.json') as f:
    cargadas = json.load(f)

print(cargadas == transacciones)

True


## 3. YAML

Más legible que JSON para humanos (sin llaves ni comillas obligatorias), muy usado para archivos de configuración — ya lo viste en los workflows de GitHub Actions (Unidad 6).

```python
!pip install -q pyyaml
import yaml

configuracion = {
    'monedas_permitidas': ['ARS', 'USD', 'EUR'],
    'monto_maximo': 100000,
    'logging': {'nivel': 'INFO', 'archivo': 'errores.log'},
}

with open('config.yaml', 'w') as f:
    yaml.dump(configuracion, f)

with open('config.yaml') as f:
    print(f.read())

with open('config.yaml') as f:
    cargada = yaml.safe_load(f)
print(cargada)
```

`yaml.safe_load` (en vez de `yaml.load`) evita ejecutar código arbitrario al leer un YAML de una fuente no confiable — es la opción recomendada.

Documentación oficial: [PyYAML](https://pyyaml.org/wiki/PyYAMLDocumentation)

In [4]:
!pip install -q pyyaml
import yaml

configuracion = {
    'monedas_permitidas': ['ARS', 'USD', 'EUR'],
    'monto_maximo': 100000,
    'logging': {'nivel': 'INFO', 'archivo': 'errores.log'},
}

with open('config.yaml', 'w') as f:
    yaml.dump(configuracion, f)

with open('config.yaml') as f:
    print(f.read())

with open('config.yaml') as f:
    cargada = yaml.safe_load(f)
print(cargada)

logging:
  archivo: errores.log
  nivel: INFO
monedas_permitidas:
- ARS
- USD
- EUR
monto_maximo: 100000

{'logging': {'archivo': 'errores.log', 'nivel': 'INFO'}, 'monedas_permitidas': ['ARS', 'USD', 'EUR'], 'monto_maximo': 100000}


## 4. Parquet

Un formato binario y columnar, pensado para grandes volúmenes de datos analíticos: comprime mejor que CSV/JSON y es mucho más rápido de leer cuando solo necesitás algunas columnas de una tabla enorme. Es el formato típico en pipelines de datos (Spark, data lakes).

```python
!pip install -q pandas pyarrow
import pandas as pd

df = pd.DataFrame([
    {'id': 1, 'monto': 100.0, 'moneda': 'ARS'},
    {'id': 2, 'monto': 200.0, 'moneda': 'USD'},
])

df.to_parquet('transacciones.parquet')
leido = pd.read_parquet('transacciones.parquet')
print(leido)
```

| Formato | Legible por humanos | Tamaño | Uso típico |
|---|---|---|---|
| JSON | Sí | Medio | Configuración, APIs |
| CSV | Sí | Medio | Intercambio simple, planillas |
| YAML | Sí (el más legible) | Medio | Configuración |
| Parquet | No (binario) | Chico (comprimido) | Analítica, grandes volúmenes |

Documentación oficial: [pandas.read_parquet](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html) · [Apache Parquet](https://parquet.apache.org/docs/)

In [5]:
!pip install -q pandas pyarrow
import pandas as pd

df = pd.DataFrame([
    {'id': 1, 'monto': 100.0, 'moneda': 'ARS'},
    {'id': 2, 'monto': 200.0, 'moneda': 'USD'},
])

df.to_parquet('transacciones.parquet')
leido = pd.read_parquet('transacciones.parquet')
print(leido)

   id  monto moneda
0   1  100.0    ARS
1   2  200.0    USD


### Ejercicio 2 — Elegir el formato

Para cada caso, indicá qué formato usarías (JSON, CSV, YAML o Parquet) y por qué:

(a) El archivo de configuración de un workflow de GitHub Actions.

(b) Un dataset de 5 millones de filas de transacciones históricas para análisis.

(c) Exportar una lista de productos para que alguien la abra en Excel.

<details>
<summary>💡 Ver solución</summary>

(a) YAML: es el formato que usa GitHub Actions, y es el más legible para configuración escrita a mano.

(b) Parquet: por el volumen, la compresión y la velocidad de lectura columnar son claves.

(c) CSV: es lo que Excel abre de forma más directa y simple.

</details>

In [6]:
import csv
import json
from pathlib import Path
import pandas as pd
import yaml

# =====================================================================
# Caso (a): YAML -> Configuración legible para GitHub Actions
# =====================================================================
workflow_config = {
    "name": "CI Pipeline",
    "on": ["push", "pull_request"],
    "jobs": {
        "test": {
            "runs-on": "ubuntu-latest",
            "steps": [
                {"uses": "actions/checkout@v3"},
                {"name": "Set up Python", "uses": "actions/setup-python@v4", "with": {"python-version": "3.11"}},
                {"name": "Install dependencies", "run": "pip install -r requirements.txt"},
                {"name": "Run tests", "run": "pytest"}
            ]
        }
    }
}

path_yaml = Path("ci_workflow.yml")
with open(path_yaml, "w", encoding="utf-8") as f:
    yaml.dump(workflow_config, f, sort_keys=False, default_flow_style=False)

print(f"✔ Caso (a) generado: {path_yaml.name}")
print(path_yaml.read_text(encoding="utf-8")[:180] + "...\n")


# =====================================================================
# Caso (b): Parquet -> Almacenamiento columnar eficiente para grandes volúmenes
# =====================================================================
# Simulación representativa del dataset transaccional
datos_transacciones = {
    "id_transaccion": range(10001, 10006),
    "usuario_id": [101, 102, 101, 103, 104],
    "monto": [1500.50, 23000.0, 450.0, 12800.75, 990.0],
    "moneda": ["ARS", "ARS", "USD", "ARS", "USD"],
    "estado": ["aprobada", "aprobada", "rechazada", "aprobada", "pendiente"]
}

df_transacciones = pd.DataFrame(datos_transacciones)

# Guardar en formato Parquet (con compresión snappy por defecto)
path_parquet = Path("transacciones_historicas.parquet")
df_transacciones.to_parquet(path_parquet, engine="auto")

print(f"✔ Caso (b) generado: {path_parquet.name}")
df_leido = pd.read_parquet(path_parquet)
print(f"Columnas leídas ({len(df_leido)} filas):\n{df_leido.head(2)}\n")


# =====================================================================
# Caso (c): CSV -> Formato universal para abrir directamente en Excel
# =====================================================================
productos = [
    {"id": 1, "nombre": "Notebook Pro 14", "precio": 1200.0, "stock": 15},
    {"id": 2, "nombre": "Mouse Óptico USB", "precio": 25.5, "stock": 80},
    {"id": 3, "nombre": "Monitor 27 Pulgadas", "precio": 310.0, "stock": 22},
]

path_csv = Path("catalogo_productos.csv")
with open(path_csv, "w", newline="", encoding="utf-8-sig") as f:
    # utf-8-sig incluye el BOM para que Excel detecte correctamente acentos y caracteres especiales
    writer = csv.DictWriter(f, fieldnames=["id", "nombre", "precio", "stock"])
    writer.writeheader()
    writer.writerows(productos)

print(f"✔ Caso (c) generado: {path_csv.name}")
print(path_csv.read_text(encoding="utf-8-sig"))

✔ Caso (a) generado: ci_workflow.yml
name: CI Pipeline
'on':
- push
- pull_request
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v3
    - name: Set up Python
      uses: actions/setu...

✔ Caso (b) generado: transacciones_historicas.parquet
Columnas leídas (5 filas):
   id_transaccion  usuario_id    monto moneda    estado
0           10001         101   1500.5    ARS  aprobada
1           10002         102  23000.0    ARS  aprobada

✔ Caso (c) generado: catalogo_productos.csv
id,nombre,precio,stock
1,Notebook Pro 14,1200.0,15
2,Mouse Óptico USB,25.5,80
3,Monitor 27 Pulgadas,310.0,22



## 5. Aplicación práctica: módulo procesador de transacciones

Vamos a construir un módulo que integra todo lo visto en la Unidad 2:

- **Excepciones personalizadas** (Colab 1) para validar cada transacción.
- **Logging estructurado** de los errores encontrados, en vez de que el programa se detenga.
- **Lectura desde CSV** y **escritura de resultados a Parquet** (las transacciones válidas) y a **JSON Lines** (el log de errores).

In [7]:
%%writefile procesador_transacciones.py
"""Modulo procesador de transacciones con validacion y logging de errores."""
import csv
import json
import logging
from dataclasses import dataclass

logging.basicConfig(
    filename='errores.log',
    level=logging.ERROR,
    format='%(asctime)s %(levelname)s %(message)s',
)
logger = logging.getLogger('procesador_transacciones')

MONEDAS_VALIDAS = {'ARS', 'USD', 'EUR'}


class MontoInvalidoError(Exception):
    """Se lanza cuando el monto de una transaccion no es valido."""


class MonedaInvalidaError(Exception):
    """Se lanza cuando la moneda de una transaccion no esta permitida."""


@dataclass
class Transaccion:
    id: int
    monto: float
    moneda: str


def validar_transaccion(t: Transaccion) -> None:
    """Valida una transaccion.

    Args:
        t: la transaccion a validar.

    Raises:
        MontoInvalidoError: si el monto no es mayor a cero.
        MonedaInvalidaError: si la moneda no esta en MONEDAS_VALIDAS.
    """
    if t.monto <= 0:
        raise MontoInvalidoError(f'Transaccion {t.id}: monto {t.monto} invalido')
    if t.moneda not in MONEDAS_VALIDAS:
        raise MonedaInvalidaError(f'Transaccion {t.id}: moneda {t.moneda} invalida')


def procesar_csv(ruta_entrada: str) -> list:
    """Lee transacciones desde un CSV, valida cada una y loguea los errores.

    Args:
        ruta_entrada: ruta al archivo CSV con columnas id, monto, moneda.

    Returns:
        La lista de transacciones validas como objetos Transaccion.
    """
    validas = []
    with open(ruta_entrada) as f:
        reader = csv.DictReader(f)
        for fila in reader:
            t = Transaccion(id=int(fila['id']), monto=float(fila['monto']), moneda=fila['moneda'])
            try:
                validar_transaccion(t)
                validas.append(t)
            except (MontoInvalidoError, MonedaInvalidaError) as e:
                logger.error(str(e))
    return validas

Writing procesador_transacciones.py


In [9]:
from dataclasses import dataclass
import csv
import importlib
import logging
import sys

# =====================================================================
# 1. Creación del módulo procesador_transacciones.py
# =====================================================================
with open("procesador_transacciones.py", "w", encoding="utf-8") as f:
    f.write('''import csv
import logging
from dataclasses import dataclass

@dataclass
class Transaccion:
    id: int
    monto: float
    moneda: str

# Configuración del logger con FileHandler explícito para Jupyter/Colab
logger = logging.getLogger("procesador_transacciones")
logger.setLevel(logging.ERROR)
logger.handlers.clear()  # Evita duplicar logs si se vuelve a ejecutar la celda

formatter = logging.Formatter("ERROR:%(name)s:%(message)s")

# 1. Handler para guardar en errores.log
file_handler = logging.FileHandler("errores.log", mode="w", encoding="utf-8")
file_handler.setLevel(logging.ERROR)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# 2. Handler para emitir también por consola
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.ERROR)
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)


def procesar_csv(ruta_csv: str) -> list[Transaccion]:
    monedas_validas = {"ARS", "USD", "EUR"}
    transacciones_validas = []

    with open(ruta_csv, mode="r", encoding="utf-8") as archivo:
        lector = csv.DictReader(archivo)
        for fila in lector:
            tx_id = int(fila["id"])
            monto = float(fila["monto"])
            moneda = fila["moneda"].strip().upper()

            if monto <= 0:
                logger.error(f"Transaccion {tx_id}: monto {monto} invalido")
                continue

            if moneda not in monedas_validas:
                logger.error(f"Transaccion {tx_id}: moneda {moneda} invalida")
                continue

            transacciones_validas.append(
                Transaccion(id=tx_id, monto=monto, moneda=moneda)
            )

    return transacciones_validas
''')

# Recargar el módulo por si ya estaba cargado en caché
if "procesador_transacciones" in sys.modules:
    importlib.reload(sys.modules["procesador_transacciones"])

# =====================================================================
# 2. Ejecución de la prueba
# =====================================================================
csv_prueba = """id,monto,moneda
1,100.0,ARS
2,-50.0,USD
3,75.0,GBP
4,300.0,EUR
"""

with open("transacciones_entrada.csv", "w", encoding="utf-8") as f:
    f.write(csv_prueba)

import procesador_transacciones as pt

validas = pt.procesar_csv("transacciones_entrada.csv")
print(f"{len(validas)} transacciones validas de 4")
for t in validas:
    print(t)

print("\n--- Contenido de errores.log ---")
with open("errores.log", "r", encoding="utf-8") as f:
    print(f.read().strip())

ERROR:procesador_transacciones:Transaccion 2: monto -50.0 invalido
ERROR:procesador_transacciones:Transaccion 2: monto -50.0 invalido
ERROR:procesador_transacciones:Transaccion 3: moneda GBP invalida
ERROR:procesador_transacciones:Transaccion 3: moneda GBP invalida


2 transacciones validas de 4
Transaccion(id=1, monto=100.0, moneda='ARS')
Transaccion(id=4, monto=300.0, moneda='EUR')

--- Contenido de errores.log ---
ERROR:procesador_transacciones:Transaccion 2: monto -50.0 invalido
ERROR:procesador_transacciones:Transaccion 3: moneda GBP invalida


### Ejercicio 3 — Guardar las transacciones válidas

Agregá al módulo (o escribí acá directamente) una función `guardar_parquet(transacciones: list, ruta: str) -> None` que convierta la lista de `Transaccion` a un `DataFrame` de pandas y la guarde como Parquet.

<details>
<summary>💡 Ver solución</summary>

```python
import pandas as pd

def guardar_parquet(transacciones, ruta):
    df = pd.DataFrame([{'id': t.id, 'monto': t.monto, 'moneda': t.moneda} for t in transacciones])
    df.to_parquet(ruta)

guardar_parquet(validas, 'transacciones_validas.parquet')
print(pd.read_parquet('transacciones_validas.parquet'))
```

</details>

In [10]:
import pandas as pd

def guardar_parquet(transacciones, ruta):
    df = pd.DataFrame([{'id': t.id, 'monto': t.monto, 'moneda': t.moneda} for t in transacciones])
    df.to_parquet(ruta)

guardar_parquet(validas, 'transacciones_validas.parquet')
print(pd.read_parquet('transacciones_validas.parquet'))

   id  monto moneda
0   1  100.0    ARS
1   4  300.0    EUR


### Ejercicio 4 — Log en formato JSON Lines

En vez de un log de texto plano, muchos sistemas prefieren JSON Lines (un JSON por línea) porque es más fácil de procesar después. Modificá la función `validar_transaccion` (o escribí una versión nueva) para que, en vez de `logger.error(str(e))`, escriba una línea JSON a un archivo `errores.jsonl` con `{'id': t.id, 'error': str(e), 'tipo': type(e).__name__}`.

<details>
<summary>💡 Ver solución</summary>

```python
import json

def procesar_csv_v2(ruta_entrada, ruta_errores='errores.jsonl'):
    validas = []
    with open(ruta_entrada) as f, open(ruta_errores, 'w') as log:
        reader = pt.csv.DictReader(f)
        for fila in reader:
            t = pt.Transaccion(id=int(fila['id']), monto=float(fila['monto']), moneda=fila['moneda'])
            try:
                pt.validar_transaccion(t)
                validas.append(t)
            except (pt.MontoInvalidoError, pt.MonedaInvalidaError) as e:
                log.write(json.dumps({'id': t.id, 'error': str(e), 'tipo': type(e).__name__}) + '\n')
    return validas

procesar_csv_v2('transacciones_entrada.csv')
with open('errores.jsonl') as f:
    print(f.read())
```

</details>

In [11]:
import json

def procesar_csv_v2(ruta_entrada, ruta_errores='errores.jsonl'):
    validas = []
    with open(ruta_entrada) as f, open(ruta_errores, 'w') as log:
        reader = pt.csv.DictReader(f)
        for fila in reader:
            t = pt.Transaccion(id=int(fila['id']), monto=float(fila['monto']), moneda=fila['moneda'])
            try:
                pt.validar_transaccion(t)
                validas.append(t)
            except (pt.MontoInvalidoError, pt.MonedaInvalidaError) as e:
                log.write(json.dumps({'id': t.id, 'error': str(e), 'tipo': type(e).__name__}) + '\n')
    return validas

procesar_csv_v2('transacciones_entrada.csv')
with open('errores.jsonl') as f:
    print(f.read())

{"id": 2, "error": "Transaccion 2: monto -50.0 invalido", "tipo": "MontoInvalidoError"}
{"id": 3, "error": "Transaccion 3: moneda GBP invalida", "tipo": "MonedaInvalidaError"}



## Mini-proyecto final: pipeline completo

Extendé `procesador_transacciones.py` para que sea un pipeline de punta a punta:

1. `procesar_csv(ruta_entrada)` — como ya está, valida y separa transacciones válidas/inválidas.
2. `guardar_parquet(transacciones, ruta)` — guarda las válidas.
3. `guardar_log_errores(errores, ruta)` — guarda los errores en JSON Lines (uno por línea).
4. `generar_reporte(validas, errores) -> dict` — devuelve un resumen: `{'total': N, 'validas': X, 'invalidas': Y, 'tasa_error': Y/N}`.
5. Un bloque `if __name__ == '__main__':` que corra las 4 funciones en orden sobre un archivo de entrada, y muestre el reporte final.

**Entregable:** el módulo `procesador_transacciones.py` completo + el archivo Parquet + el log de errores + el reporte impreso, corridos sobre al menos 10 transacciones (con al menos 3 inválidas).

---

**Fin de la Unidad 2.** Con estos tres notebooks recorriste el ciclo completo: manejar errores de forma prolija, organizar el código en módulos/paquetes con entornos aislados, y persistir datos en distintos formatos — la base de todo lo que construiste en las Unidades 3, 4, 5 y 6.

In [12]:
import csv
from dataclasses import asdict, dataclass
import json
from pathlib import Path
import sys
import pandas as pd

# =====================================================================
# 1. Creación del módulo procesador_transacciones.py
# =====================================================================
with open("procesador_transacciones.py", "w", encoding="utf-8") as f:
    f.write('''from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple, Any
import csv
import json
import pandas as pd


@dataclass
class Transaccion:
    id: int
    monto: float
    moneda: str


def procesar_csv(ruta_entrada: str) -> Tuple[List[Transaccion], List[Dict[str, Any]]]:
    """Valida el archivo CSV y separa transacciones válidas de inválidas."""
    monedas_validas = {"ARS", "USD", "EUR"}
    validas: List[Transaccion] = []
    errores: List[Dict[str, Any]] = []

    with open(ruta_entrada, mode="r", encoding="utf-8") as archivo:
        lector = csv.DictReader(archivo)
        for fila in lector:
            tx_id_raw = fila.get("id", "").strip()
            monto_raw = fila.get("monto", "").strip()
            moneda_raw = fila.get("moneda", "").strip().upper()

            try:
                tx_id = int(tx_id_raw)
            except ValueError:
                errores.append({
                    "id": tx_id_raw,
                    "monto": monto_raw,
                    "moneda": moneda_raw,
                    "motivo": f"ID no numérico: '{tx_id_raw}'"
                })
                continue

            try:
                monto = float(monto_raw)
            except ValueError:
                errores.append({
                    "id": tx_id,
                    "monto": monto_raw,
                    "moneda": moneda_raw,
                    "motivo": f"Monto no convertible a float: '{monto_raw}'"
                })
                continue

            if monto <= 0:
                errores.append({
                    "id": tx_id,
                    "monto": monto,
                    "moneda": moneda_raw,
                    "motivo": f"Monto {monto} inválido (debe ser mayor a 0)"
                })
                continue

            if moneda_raw not in monedas_validas:
                errores.append({
                    "id": tx_id,
                    "monto": monto,
                    "moneda": moneda_raw,
                    "motivo": f"Moneda '{moneda_raw}' inválida (admitidas: {sorted(list(monedas_validas))})"
                })
                continue

            validas.append(Transaccion(id=tx_id, monto=monto, moneda=moneda_raw))

    return validas, errores


def guardar_parquet(transacciones: List[Transaccion], ruta: str) -> None:
    """Guarda la lista de transacciones válidas en formato Parquet."""
    if not transacciones:
        df = pd.DataFrame(columns=["id", "monto", "moneda"])
    else:
        df = pd.DataFrame([asdict(t) for t in transacciones])
    df.to_parquet(ruta, index=False, engine="auto")


def guardar_log_errores(errores: List[Dict[str, Any]], ruta: str) -> None:
    """Guarda las transacciones fallidas en formato JSON Lines (una por línea)."""
    with open(ruta, mode="w", encoding="utf-8") as f:
        for error in errores:
            f.write(json.dumps(error, ensure_ascii=False) + "\\n")


def generar_reporte(validas: List[Transaccion], errores: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Genera un resumen métrico del pipeline."""
    total = len(validas) + len(errores)
    cant_validas = len(validas)
    cant_invalidas = len(errores)
    tasa_error = (cant_invalidas / total) if total > 0 else 0.0

    return {
        "total": total,
        "validas": cant_validas,
        "invalidas": cant_invalidas,
        "tasa_error": round(tasa_error, 4)
    }


if __name__ == "__main__":
    archivo_csv = "transacciones_entrada.csv"
    salida_parquet = "transacciones_validas.parquet"
    salida_errores = "errores_transacciones.jsonl"

    print("--> Iniciando pipeline de procesamiento...")
    tx_validas, tx_errores = procesar_csv(archivo_csv)

    print(f"--> Exportando {len(tx_validas)} transacciones válidas a Parquet...")
    guardar_parquet(tx_validas, salida_parquet)

    print(f"--> Guardando {len(tx_errores)} errores en log JSON Lines...")
    guardar_log_errores(tx_errores, salida_errores)

    reporte = generar_reporte(tx_validas, tx_errores)
    print("\\n" + "=" * 55)
    print("                 REPORTE FINAL")
    print("=" * 55)
    print(json.dumps(reporte, indent=2, ensure_ascii=False))
    print("=" * 55)
''')

# =====================================================================
# 2. Generación del dataset de entrada (11 transacciones, 4 inválidas)
# =====================================================================
csv_datos = """id,monto,moneda
1,1500.50,ARS
2,-250.00,USD
3,89.99,EUR
4,0.00,ARS
5,12000.00,ARS
6,45.50,GBP
7,350.00,USD
8,-15.00,EUR
9,740.20,ARS
10,999.00,USD
11,54.00,BRL
"""

with open("transacciones_entrada.csv", "w", encoding="utf-8") as f:
    f.write(csv_datos)

# =====================================================================
# 3. Ejecución del pipeline principal
# =====================================================================
# Ejecución directa del módulo como script principal (__main__)
if "procesador_transacciones" in sys.modules:
    del sys.modules["procesador_transacciones"]

import procesador_transacciones as pt

# Ejecución de las 4 funciones en secuencia
tx_validas, tx_errores = pt.procesar_csv("transacciones_entrada.csv")
pt.guardar_parquet(tx_validas, "transacciones_validas.parquet")
pt.guardar_log_errores(tx_errores, "errores_transacciones.jsonl")
reporte_final = pt.generar_reporte(tx_validas, tx_errores)

# =====================================================================
# 4. Muestra de los archivos resultantes y el reporte
# =====================================================================
print("=" * 65)
print("1. REPORTE FINAL GENERADO")
print("=" * 65)
print(json.dumps(reporte_final, indent=2, ensure_ascii=False))

print("\n" + "=" * 65)
print("2. CONTENIDO DEL ARCHIVO PARQUET (Transacciones Válidas)")
print("=" * 65)
df_parquet = pd.read_parquet("transacciones_validas.parquet")
print(df_parquet.to_string(index=False))

print("\n" + "=" * 65)
print("3. CONTENIDO DEL ARCHIVO JSONL (Log de Errores)")
print("=" * 65)
with open("errores_transacciones.jsonl", "r", encoding="utf-8") as f:
    print(f.read().strip())

1. REPORTE FINAL GENERADO
{
  "total": 11,
  "validas": 6,
  "invalidas": 5,
  "tasa_error": 0.4545
}

2. CONTENIDO DEL ARCHIVO PARQUET (Transacciones Válidas)
 id    monto moneda
  1  1500.50    ARS
  3    89.99    EUR
  5 12000.00    ARS
  7   350.00    USD
  9   740.20    ARS
 10   999.00    USD

3. CONTENIDO DEL ARCHIVO JSONL (Log de Errores)
{"id": 2, "monto": -250.0, "moneda": "USD", "motivo": "Monto -250.0 inválido (debe ser mayor a 0)"}
{"id": 4, "monto": 0.0, "moneda": "ARS", "motivo": "Monto 0.0 inválido (debe ser mayor a 0)"}
{"id": 6, "monto": 45.5, "moneda": "GBP", "motivo": "Moneda 'GBP' inválida (admitidas: ['ARS', 'EUR', 'USD'])"}
{"id": 8, "monto": -15.0, "moneda": "EUR", "motivo": "Monto -15.0 inválido (debe ser mayor a 0)"}
{"id": 11, "monto": 54.0, "moneda": "BRL", "motivo": "Moneda 'BRL' inválida (admitidas: ['ARS', 'EUR', 'USD'])"}
